# Parallel-groups lab

Extract one page's raw vectors and cluster them exactly as P3 **VectorClassification** does: layer -> color ->
stroke-width buckets -> seqno-overlap merge -> constrained spatial clustering.

Then, **within each cluster**, take the **straight** Vectors (all items `'l'`, every point within
`STRAIGHT_TOL_PT` of one line) and group them by **direction**: angle within `ANGLE_TOL_DEG`, folded to
[0, 180), at any offset.

Layers (open with `scripts/pipeline_report_viewer.py`):
- `parallel / angle`: HSV colour with saturation 100 %.
  - **Hue = cluster.**
  - **Value = group angle**, as a triangle wave: 0/180 deg -> 25 %, 90 deg -> 75 %. So 45 deg and 135 deg
    share a value.
- `gradient / vector count` and `gradient / length std`: green = lowest ... red = highest.
- `other / singletons`, `other / excluded (non-straight)` and `clusters / cluster bboxes`: context.

Gradient ranges are page-wide and only cover groups with >= 2 members.

## 0 - Config

In [ ]:
PDF_PATH = None         # None -> first PDF under references/
PAGE = 0
ANGLE_TOL_DEG = 1.0     # single-linkage angle tolerance (folded to [0, 180))
STRAIGHT_TOL_PT = 0.25  # max point distance from the fitted line for a Vector to count as straight
LINE_WIDTH = 1.0

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + cluster (P3 VectorClassification chain)

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
clusters = h.cluster_vectors(vectors, page_meta)
print(f'{len(vectors)} vectors -> {len(clusters)} clusters')

## 3 - Parallel groups within each cluster

In [ ]:
groups, group_cluster, excluded = [], [], []
for ci, (key, members) in enumerate(clusters):
    straight, rest = h.split_straight(members, STRAIGHT_TOL_PT)
    excluded.extend(rest)
    for g in h.group_parallel(straight, ANGLE_TOL_DEG):
        groups.append(g)
        group_cluster.append(ci)

In [ ]:
multi = [g for g in groups if len(g) >= 2]
singles = [g for g in groups if len(g) == 1]
stats = [h.group_stats(g) for g in multi]
counts = [c for c, _ in stats]
stds = [s for _, s in stats]
c_lo, c_hi = (min(counts), max(counts)) if counts else (0, 0)
s_lo, s_hi = (min(stds), max(stds)) if stds else (0.0, 0.0)
print(f'{len(groups)} group(s): {len(multi)} with >=2 members, {len(singles)} singleton(s); '
      f'{len(excluded)} non-straight vector(s) excluded')
print(f'count range {c_lo}..{c_hi}, length-std range {s_lo:.3f}..{s_hi:.3f} pt')

## 4 - Debug layers

In [ ]:
report = h.DebugReport('parallel_groups_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)
hues = h.golden_hues(len(clusters))

angle_colour, angle_vecs = {}, []
for g, ci in zip(groups, group_cluster):
    if len(g) < 2:
        continue
    rgb = h.hsv_cluster_angle(hues[ci], h._mean_axis_angle([fit.angle for _, fit in g]))
    for v, _ in g:
        angle_colour[id(v)] = rgb
        angle_vecs.append(v)
report.add_vectors('parallel', 'angle (hue=cluster, value=angle)', h.hsv_cluster_angle(0.6, 90),
                   angle_vecs, lambda v: angle_colour[id(v)])

In [ ]:
def _gradient_layer(label, values, lo, hi):
    """One layer: every multi-member group's vectors coloured green (lo)
    -> red (hi), drawn low-to-high so the extreme groups end up on top."""
    order = sorted(range(len(multi)), key=lambda i: values[i])
    colour, vecs = {}, []
    for i in order:
        rgb = h.green_red(h.normalise(values[i], lo, hi))
        for v, _ in multi[i]:
            colour[id(v)] = rgb
            vecs.append(v)
    report.add_vectors('gradient', label, h.green_red(1.0), vecs, lambda v: colour[id(v)])

_gradient_layer(f'vector count ({c_lo}-{c_hi})', counts, c_lo, c_hi)
_gradient_layer(f'length std ({s_lo:.2f}-{s_hi:.2f} pt)', stds, s_lo, s_hi)

report.add_vectors('other', 'singletons', h.GREY, [v for g in singles for v, _ in g], lambda v: h.GREY)
report.add_vectors('other', 'excluded (non-straight)', h.LIGHT_GREY, excluded, lambda v: h.LIGHT_GREY)

report.add_boxes('clusters', 'cluster bboxes', '#888888', [
    (union_bbox([v.bbox for v in members]), colorsys.hsv_to_rgb(hues[ci], 1.0, 0.6))
    for ci, (_, members) in enumerate(clusters)
])

## 5 - Histograms

In [ ]:
GROUP_NAME = 'Parallel'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if counts:
    size_bins = np.arange(2, max(counts) + 2) - 0.5 if max(counts) < 60 else 40
    axes[0].hist(counts, bins=size_bins, color='#4a7bd0', edgecolor='white')
axes[0].set_title(f'{GROUP_NAME} group size (>=2 members; {len(singles)} singletons not shown)')
axes[0].set_xlabel('vectors in group')
axes[0].set_ylabel('groups')
if stds:
    axes[1].hist(stds, bins=40, color='#d07a4a', edgecolor='white')
axes[1].set_title(f'{GROUP_NAME} group length std (>=2 members)')
axes[1].set_xlabel('population std of vector lengths (pt)')
axes[1].set_ylabel('groups')
fig.tight_layout()
report.save_figure(fig, 'histograms')
plt.show()

## 6 - Top 15 groups by member count

In [ ]:
multi_cluster = [ci for g, ci in zip(groups, group_cluster) if len(g) >= 2]
rows = sorted(zip(multi, stats, multi_cluster), key=lambda r: -r[1][0])[:15]
print(f"{'cluster':>7} {'n':>5} {'angle':>7} {'mean len':>9} {'len std':>8}")
for g, (n, std), ci in rows:
    angle = h._mean_axis_angle([fit.angle for _, fit in g])
    print(f'{ci:>7} {n:>5} {angle:>7.2f} {np.mean([f.length for _, f in g]):>9.2f} {std:>8.3f}')

## 7 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())